# CellPhoneDB Method 3 (DEG-based) — Cluster-Cluster & Per-Cell Interaction Score

## 분석 흐름
```
sctotal_01.h5ad  +  cellseq_total_cellphneDB.tsv
        │
        ▼
[1] CellPhoneDB Method 3  (cpdb_degs_analysis_method)
        │
        ├──▶ [2] Cluster-Cluster interaction heatmap
        ├──▶ [3] Per-cell score  →  adata.obs['clusterA__LR1__score']
        └──▶ [4] Save adata + tables
```

**Per-cell score 명명 규칙**  
`{sender_cluster}__{LR_pair}__score`  
- 해당 클러스터에 속하는 세포만 값 부여, 나머지는 `NaN`  
- `score` = 모든 partner 클러스터 대상 interaction score 중 **최댓값**  
- 컬럼명 특수문자(` | - /`)는 `_`로 치환

## 0. Import & Path 설정

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import os
import re
import pickle

from cellphonedb.src.core.methods import cpdb_degs_analysis_method
import ktplotspy as kpy

# ── font 설정 (PDF 저장 시 폰트 깨짐 방지) ────────────────────────────────────
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42
mpl.rcParams['svg.fonttype'] = 'none'

sc.settings.verbosity = 1
print('scanpy:', sc.__version__)

In [ ]:
adata=sc.read("/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_cellphoneDB.h5ad")
adata

In [ ]:
adata.obs['cluster_state'] = adata.obs['detail_cluster'].astype(str) + '-' + adata.obs['state'].astype(str)
adata.obs['cluster_state'].value_counts()

In [ ]:
adata.write("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/01_Data/sctotal_01.h5ad")

In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────────
BASE      = '/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal'
DATA_DIR  = os.path.join(BASE, '01_Data')
ANAL_DIR  = os.path.join(BASE, '02_Analysis')
FIG_DIR   = os.path.join(BASE, '05_Figure')
CPDB_DIR  = os.path.join(ANAL_DIR, 'cellphoneDB')
OUT_DIR   = os.path.join(CPDB_DIR, 'method3')

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ── CellPhoneDB 입력 파일 경로 ────────────────────────────────────────────────
CPDB_ZIP   = '/home/jaehyunchoi/Tool_testing/cellphoneDB/db/test/v5.0.0/cellphonedb.zip'
META_TSV   = os.path.join(CPDB_DIR, 'cellseq_total_cellphneDB.tsv')
COUNTS_H5AD = '/home/jaehyunchoi/Liver_bio_project/MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_raw.h5ad'
DEGS_TXT   = os.path.join(CPDB_DIR, 'degs_mono3_MASLD_vs_normal.txt')

# ── 분석용 adata 경로 (cluster_state obs 컬럼이 있는 processed adata) ─────────
ADATA_PATH = os.path.join(DATA_DIR, 'sctotal_01.h5ad')


# ── 결과 저장 경로 ─────────────────────────────────────────────────────────────
PKL_PATH        = os.path.join(OUT_DIR, 'cpdb_method3_results.pkl')
ADATA_OUT       = os.path.join(DATA_DIR, 'sctotal_01_cpdb_scored.h5ad')
SCORE_CSV_OUT   = os.path.join(OUT_DIR, 'percell_cci_scores.csv')

# ── 관심 클러스터 (per-cell score를 뽑을 클러스터, None이면 전체) ─────────────
# 예: FOCUS_CLUSTERS = ['mono3-Healthy', 'mono3-MASL']
FOCUS_CLUSTERS = None

# ── metadata의 cell-type 컬럼명 ──────────────────────────────────────────────
CELLTYPE_COL = 'cluster_state'   # adata.obs에도 동일 컬럼이 있어야 함

# ── cutree (클러스터 수) ─────────────────────────────────────────────────────
N_CLUSTERS_CUTOFF = 6    # relevant_interaction 필터링 기준 (None = 필터 없이 전체 사용)

## 1. Metadata & adata 확인

In [ ]:
# ── metadata 확인 ──────────────────────────────────────────────────────────────
metadata = pd.read_csv(META_TSV, sep='\t')
print('metadata shape:', metadata.shape)
print('columns:', metadata.columns.tolist())
print()
print('cluster_state 분포:')
print(metadata[CELLTYPE_COL].value_counts())

In [ ]:
# ── DEG 파일 확인 ──────────────────────────────────────────────────────────────
degs_df = pd.read_csv(DEGS_TXT, sep='\t')
print('DEG file shape:', degs_df.shape)
print('columns:', degs_df.columns.tolist())
degs_df.head(3)

In [ ]:
# ── 분석용 adata 로딩 ─────────────────────────────────────────────────────────
adata = sc.read_h5ad(ADATA_PATH)
print(adata)
print()
print(f"'{CELLTYPE_COL}' obs 컬럼 존재:", CELLTYPE_COL in adata.obs.columns)

if CELLTYPE_COL in adata.obs.columns:
    print('\nadata cluster_state 분포:')
    print(adata.obs[CELLTYPE_COL].value_counts())

In [ ]:
# ── metadata barcode vs adata index 일치 확인 ─────────────────────────────────
meta_barcodes = set(metadata['barcode'].tolist())
adata_barcodes = set(adata.obs_names.tolist())

overlap = meta_barcodes & adata_barcodes
print(f'metadata 바코드: {len(meta_barcodes)}')
print(f'adata 바코드:     {len(adata_barcodes)}')
print(f'겹치는 바코드:    {len(overlap)}')

# metadata와 adata 인덱스가 다를 경우 adata를 metadata 순서로 subset
if len(overlap) < len(meta_barcodes):
    print('\n[주의] 일부 바코드 불일치 — overlap 세포만 사용합니다.')
    adata = adata[adata.obs_names.isin(overlap)].copy()

## 2. CellPhoneDB Method 3 실행 (DEG-based analysis)

In [ ]:
# ── Method 3: cpdb_degs_analysis_method ───────────────────────────────────────
# DEG 파일에 의해 어떤 ligand/receptor가 upregulated인지 기반으로 유의 상호작용 선별
# 기존 method1(simple), method2(statistical)과 달리 condition-specific DEG 반영

cpdb_results = cpdb_degs_analysis_method.call(
    cpdb_file_path          = CPDB_ZIP,        # CellphoneDB database zip
    meta_file_path          = META_TSV,        # 바코드 → 클러스터 매핑 tsv
    counts_file_path        = COUNTS_H5AD,     # normalized count matrix (raw h5ad)
    degs_file_path          = DEGS_TXT,        # DEG 파일 (group, names, logfoldchanges ...)
    counts_data             = 'hgnc_symbol',   # gene annotation 형식
    output_path             = OUT_DIR,         # 결과 txt 파일 저장 경로
    output_suffix           = '',
    score_interactions      = True,            # interaction_scores 계산 (필수)
    threshold               = 0.1,             # 각 클러스터에서 발현 세포 비율 threshold
    result_precision        = 3,
    debug                   = False,
)

print('완료! result keys:', list(cpdb_results.keys()))

In [ ]:
# ── 결과 pkl 저장 (재시작 포인트) ────────────────────────────────────────────
with open(PKL_PATH, 'wb') as f:
    pickle.dump(cpdb_results, f)
print('저장 완료:', PKL_PATH)

In [ ]:
# ── [재시작 시] pkl에서 결과 불러오기 ────────────────────────────────────────
# 위 Section 2 실행을 건너뛰고 싶을 때 이 셀만 실행
with open(PKL_PATH, 'rb') as f:
    cpdb_results = pickle.load(f)
print('로드 완료! keys:', list(cpdb_results.keys()))

In [ ]:
# ── 결과 구조 확인 ────────────────────────────────────────────────────────────
for k, v in cpdb_results.items():
    if hasattr(v, 'shape'):
        print(f'{k:35s}  shape={v.shape}')
    else:
        print(f'{k:35s}  type={type(v)}')

## 3. Cluster-Cluster Interaction 시각화

In [ ]:
# ── ktplotspy: 유의 interaction 수 heatmap ────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 13))
kpy.plot_cpdb_heatmap(
    pvals         = cpdb_results['relevant_interactions'],
    degs_analysis = True,
    figsize       = (15, 13),
    title         = 'Sum of significant interactions (Method 3)'
)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_interaction_heatmap.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_interaction_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── annotation 컬럼 정의 ─────────────────────────────────────────────────────
ANNOT_COLS = [
    'id_cp_interaction', 'interacting_pair', 'partner_a', 'partner_b',
    'gene_a', 'gene_b', 'secreted', 'receptor_a', 'receptor_b',
    'annotation_strategy', 'is_integrin', 'directionality', 'classification'
]

# ── interaction_scores 및 relevant_interactions 파싱 ─────────────────────────
IS = cpdb_results['interaction_scores'].copy()
RI = cpdb_results['relevant_interactions'].copy()

# interacting_pair를 인덱스로 설정
IS = IS.set_index('interacting_pair')
RI = RI.set_index('interacting_pair')

# cluster pair 컬럼 (|로 구분된 컬럼)
pair_cols = [c for c in IS.columns if c not in ANNOT_COLS and '|' in c]
print(f'총 cluster pair 수: {len(pair_cols)}')
print(f'총 LR pair 수 (interaction_scores): {len(IS)}')
print(f'총 LR pair 수 (relevant_interactions): {len(RI)}')
print('\n예시 pair columns:')
print(pair_cols[:6])

In [ ]:
# ── cluster-cluster 총합 interaction score 행렬 만들기 ────────────────────────
# 각 클러스터 pair에서 유의한 LR pair의 interaction score 합산

# relevant_interactions에서 유의한 LR pair 식별 (값 != 0)
IS_pair = IS[pair_cols].fillna(0)
RI_pair = RI[pair_cols].fillna(0)

# 유의 LR pair 필터 (relevant_interactions에서 하나 이상의 pair에서 significant)
sig_mask  = (RI_pair != 0).any(axis=1)
sig_lr_pairs = RI_pair[sig_mask].index.tolist()
print(f'유의한 LR pair 수: {len(sig_lr_pairs)}')

# 클러스터 pair별 총합 (유의 LR pair만)
cluster_pair_total = IS_pair.loc[sig_lr_pairs].sum(axis=0)   # Series: index=pair_col

# 2D 행렬로 변환 (sender x receiver)
unique_clusters = sorted(set(
    [c.split('|')[0] for c in pair_cols] + [c.split('|')[1] for c in pair_cols]
))

score_matrix = pd.DataFrame(index=unique_clusters, columns=unique_clusters, dtype=float)
score_matrix[:] = 0.0

for pair_col in pair_cols:
    sender, receiver = pair_col.split('|')
    score_matrix.loc[sender, receiver] = cluster_pair_total.get(pair_col, 0)

score_matrix = score_matrix.astype(float)
print('Score matrix shape:', score_matrix.shape)

In [ ]:
# ── 클러스터-클러스터 total score heatmap ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    score_matrix,
    cmap       = 'YlOrRd',
    linewidths = 0.3,
    linecolor  = 'white',
    ax         = ax,
    cbar_kws   = {'label': 'Total interaction score'}
)
ax.set_xlabel('Receiver cluster', fontsize=12)
ax.set_ylabel('Sender cluster',   fontsize=12)
ax.set_title('Cluster-Cluster Total Interaction Score (Method 3)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_cluster_score_heatmap.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_cluster_score_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

# CSV 저장
score_matrix.to_csv(os.path.join(OUT_DIR, 'cluster_pair_total_score.csv'))
print('cluster_pair_total_score.csv 저장 완료')

## 4. Per-Cell Interaction Score → `adata.obs`

### 명명 규칙
- `{sender_cluster}__{LR_pair}__score`  
  예) `mono3_Healthy__LRP1_LDLR__score`
- 해당 cluster에 속하는 세포만 score 부여, 나머지는 `NaN`
- score = 모든 receiver 클러스터 중 최대값 (significant interaction 기준)

### 추가 컬럼
- `CCI_sender_total_score`: 세포가 속한 클러스터의 모든 LR pair score 합산
- `CCI_receiver_total_score`: 세포가 속한 클러스터가 receiver일 때의 score 합산

In [ ]:
def sanitize_name(s):
    """obs 컬럼명으로 사용 가능하도록 특수문자 치환."""
    return re.sub(r'[|\-\s/+().,;:]', '_', str(s)).strip('_')


def assign_percell_scores(
    adata,
    IS_pair,           # interaction_scores (pair cols only, index=interacting_pair)
    RI_pair,           # relevant_interactions (pair cols only, index=interacting_pair)
    pair_cols,
    celltype_col,
    focus_clusters=None,
    agg='max'          # 'max' or 'sum' — partner 클러스터 간 aggregation 방법
):
    """
    각 세포에 CCI score를 adata.obs 컬럼으로 부여.

    Parameters
    ----------
    agg : 'max' | 'sum'
        여러 partner 클러스터 중 score를 어떻게 집계할지 결정.
    """
    cell_clusters = adata.obs[celltype_col].astype(str)
    target_clusters = focus_clusters if focus_clusters else cell_clusters.unique().tolist()

    # 유의 LR pairs: RI에서 하나 이상 significant (!=0)인 LR pair
    sig_mask     = (RI_pair != 0).any(axis=1)
    sig_lr_set   = set(RI_pair[sig_mask].index.tolist())
    sig_in_IS    = [lr for lr in sig_lr_set if lr in IS_pair.index]

    print(f'유의 LR pair: {len(sig_in_IS)}개')
    print(f'대상 cluster: {len(target_clusters)}개')
    print('per-cell score 컬럼 생성 시작...')

    # ── 총합 score 컬럼 초기화 ───────────────────────────────────────────────
    adata.obs['CCI_sender_total_score']   = np.nan
    adata.obs['CCI_receiver_total_score'] = np.nan

    new_cols = {}   # {col_name: Series} — 한꺼번에 adata.obs에 concat

    for cluster in target_clusters:
        cell_mask = cell_clusters == cluster
        cell_idx  = cell_mask[cell_mask].index
        if len(cell_idx) == 0:
            continue

        safe_cluster = sanitize_name(cluster)

        # sender 관점: 이 cluster가 왼쪽에 있는 pair columns
        sender_cols   = [c for c in pair_cols if c.split('|')[0] == cluster]
        # receiver 관점: 이 cluster가 오른쪽에 있는 pair columns
        receiver_cols = [c for c in pair_cols if c.split('|')[1] == cluster]

        # ── sender total score ───────────────────────────────────────────────
        if sender_cols:
            # relevant한 LR에 해당하는 interaction score만 합산
            RI_sender_sig = RI_pair.loc[
                RI_pair.index.isin(sig_in_IS), sender_cols
            ]
            IS_sender_sig = IS_pair.loc[
                IS_pair.index.isin(sig_in_IS), sender_cols
            ]
            # significant한 pair에만 마스킹
            masked_score  = IS_sender_sig.where(RI_sender_sig != 0, other=0.0)
            total_sender  = float(masked_score.values.sum())
            adata.obs.loc[cell_idx, 'CCI_sender_total_score'] = total_sender

        # ── receiver total score ─────────────────────────────────────────────
        if receiver_cols:
            RI_recv_sig = RI_pair.loc[
                RI_pair.index.isin(sig_in_IS), receiver_cols
            ]
            IS_recv_sig = IS_pair.loc[
                IS_pair.index.isin(sig_in_IS), receiver_cols
            ]
            masked_recv  = IS_recv_sig.where(RI_recv_sig != 0, other=0.0)
            total_recv   = float(masked_recv.values.sum())
            adata.obs.loc[cell_idx, 'CCI_receiver_total_score'] = total_recv

        # ── per LR pair score ────────────────────────────────────────────────
        if sender_cols:
            for lr in sig_in_IS:
                if lr not in IS_pair.index:
                    continue

                # relevant (significant)한 partner들만 대상
                if lr in RI_pair.index:
                    sig_partners = [
                        c for c in sender_cols
                        if RI_pair.loc[lr, c] != 0
                    ]
                else:
                    sig_partners = sender_cols

                if not sig_partners:
                    continue

                scores = IS_pair.loc[lr, sig_partners].astype(float)
                agg_score = scores.max() if agg == 'max' else scores.sum()

                if pd.isna(agg_score):
                    continue

                safe_lr   = sanitize_name(lr)
                col_name  = f'{safe_cluster}__{safe_lr}__score'

                if col_name not in new_cols:
                    new_cols[col_name] = pd.Series(np.nan, index=adata.obs_names)
                new_cols[col_name].loc[cell_idx] = agg_score

    # ── 모아서 한 번에 concat ────────────────────────────────────────────────
    if new_cols:
        new_df = pd.DataFrame(new_cols, index=adata.obs_names)
        adata.obs = pd.concat([adata.obs, new_df], axis=1)

    n_new = len(new_cols)
    print(f'완료: {n_new}개 per-cell score 컬럼 추가')
    print('  + CCI_sender_total_score')
    print('  + CCI_receiver_total_score')
    return adata

In [ ]:
# 1. sig_in_IS 확인
sig_mask   = (RI_pair != 0).any(axis=1)
sig_lr_set = set(RI_pair[sig_mask].index)
sig_in_IS  = [lr for lr in sig_lr_set if lr in IS_pair.index]
print(f"sig_in_IS 개수: {len(sig_in_IS)}")
print(f"RI_pair 0 아닌 값 수: {sig_mask.sum()}")

# 2. pair_cols vs cluster명 매칭 확인
print("pair_cols 예시:", pair_cols[:5])
print("focus_clusters 예시:", FOCUS_CLUSTERS[:3] if FOCUS_CLUSTERS else adata.obs[CELLTYPE_COL].unique()[:3])

# 3. index 형식 일치 여부
print("IS_pair.index[:3]:", IS_pair.index[:3].tolist())
print("RI_pair.index[:3]:", RI_pair.index[:3].tolist())

In [ ]:
# assign_percell_scores 호출 전에
pair_cols_fixed = [c.lower() for c in pair_cols]
IS_pair_fixed   = IS_pair.copy()
RI_pair_fixed   = RI_pair.copy()
IS_pair_fixed.columns = [c.lower() for c in IS_pair.columns]
RI_pair_fixed.columns = [c.lower() for c in RI_pair.columns]

adata = assign_percell_scores(
    adata          = adata,
    IS_pair        = IS_pair_fixed,
    RI_pair        = RI_pair_fixed,
    pair_cols      = pair_cols_fixed,
    celltype_col   = CELLTYPE_COL,
    focus_clusters = FOCUS_CLUSTERS,
    agg            = 'max'
)

# 추가된 컬럼 확인
cci_cols = [c for c in adata.obs.columns if c.startswith('CCI_') or c.endswith('__score')]
print(f'\n총 CCI 관련 obs 컬럼: {len(cci_cols)}개')
print('예시 컬럼명:')
print(cci_cols[:10])

In [ ]:
# ── 추가된 score 컬럼 요약 통계 ───────────────────────────────────────────────
cci_score_summary = adata.obs[cci_cols].describe().T
cci_score_summary = cci_score_summary[cci_score_summary['count'] > 0]  # NaN만 있는 행 제외
print(f'실제 값이 있는 컬럼: {len(cci_score_summary)}개')
cci_score_summary.sort_values('mean', ascending=False).head(20)

## 5. Per-Cell Score 시각화 (UMAP)

In [ ]:
# ── UMAP: CCI_sender_total_score ─────────────────────────────────────────────
if 'X_umap' in adata.obsm:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc.pl.umap(
        adata, color=CELLTYPE_COL,
        legend_loc='on data', legend_fontsize=6,
        frameon=False, size=5, title='Cell type',
        ax=axes[0], show=False
    )
    sc.pl.umap(
        adata, color='CCI_sender_total_score',
        cmap='YlOrRd', frameon=False, size=5,
        title='CCI sender total score',
        ax=axes[1], show=False
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_umap_total_score.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_umap_total_score.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[주의] adata에 UMAP 좌표 없음 — sc.tl.umap() 먼저 실행 필요')

In [ ]:
# ── UMAP: CCI_sender_total_score ─────────────────────────────────────────────
if 'X_umap' in adata.obsm:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc.pl.umap(
        adata, color=CELLTYPE_COL,
        legend_loc='on data', legend_fontsize=6,
        frameon=False, size=5, title='Cell type',
        ax=axes[0], show=False
    )
    sc.pl.umap(
        adata, color='CCI_receiver_total_score',
        cmap='YlOrRd', frameon=False, size=5,
        title='CCI sender total score',
        ax=axes[1], show=False
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_umap_total_score.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_umap_total_score.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('[주의] adata에 UMAP 좌표 없음 — sc.tl.umap() 먼저 실행 필요')

In [ ]:
# ── UMAP: 특정 LR pair score 시각화 예시 ─────────────────────────────────────
# 관심 LR pair를 score_matrix에서 상위 n개로 자동 선택하거나 직접 지정

# score가 있는 컬럼 중 상위 mean 값 6개 자동 선택
top_lr_cols = (
    adata.obs[cci_cols]
    .drop(columns=['CCI_sender_total_score', 'CCI_receiver_total_score'], errors='ignore')
    .mean()
    .nlargest(6)
    .index.tolist()
)

print('시각화할 top LR score 컬럼:')
for c in top_lr_cols:
    print(' ', c)

if 'X_umap' in adata.obsm and top_lr_cols:
    sc.pl.umap(
        adata,
        color   = top_lr_cols,
        cmap    = 'YlOrRd',
        frameon = False,
        size    = 5,
        ncols   = 3
    )
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_umap_top_lr_scores.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_umap_top_lr_scores.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 클러스터별 total score boxplot ────────────────────────────────────────────
plot_df = adata.obs[[CELLTYPE_COL, 'CCI_sender_total_score']].dropna()

cluster_order = (
    plot_df.groupby(CELLTYPE_COL)['CCI_sender_total_score']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(max(8, len(cluster_order) * 0.6), 5))
sns.boxplot(
    data   = plot_df,
    x      = CELLTYPE_COL,
    y      = 'CCI_sender_total_score',
    order  = cluster_order,
    palette= 'Set2',
    ax     = ax
)
ax.set_title('CCI sender total score per cluster')
ax.set_xlabel('')
ax.set_ylabel('Total interaction score')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_cluster_score_boxplot.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_cluster_score_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. `significant_means` 기반 Healthy vs MASL Delta 분석

CellPhoneDB가 출력하는 `significant_means`는 각 cluster-pair별 LR interaction mean expression이다.  
여기서 **MASL - Healthy delta**를 계산해 질환 상태에서 증가/감소한 상호작용을 시각화한다.

In [ ]:
# ── significant_means 추출 ────────────────────────────────────────────────
sig_means = cpdb_results['significant_means'].copy()

ANNOT_COLS_SM = [
    'id_cp_interaction', 'interacting_pair', 'partner_a', 'partner_b',
    'gene_a', 'gene_b', 'secreted', 'receptor_a', 'receptor_b',
    'annotation_strategy', 'is_integrin', 'directionality', 'classification'
]

sm_pair_cols = [c for c in sig_means.columns if '|' in c]
print(f'significant_means shape: {sig_means.shape}')
print(f'cluster pair 수: {len(sm_pair_cols)}')
sig_means.head(3)

In [ ]:
# ── 상태(Healthy / MASL) 분리 helper ────────────────────────────────────────
def get_state(cell_name):
    if 'MASL' in cell_name:
        return 'MASL'
    return 'Healthy'

def clean_name(name):
    return name.replace('-Healthy', '').replace('-MASL', '')

# long-form 변환
sm_long = sig_means[['interacting_pair'] + sm_pair_cols].melt(
    id_vars='interacting_pair',
    value_vars=sm_pair_cols,
    var_name='pair',
    value_name='weight'
).dropna(subset=['weight'])

sm_long[['celltype1', 'celltype2']] = sm_long['pair'].str.split('|', expand=True)
sm_long['state1'] = sm_long['celltype1'].apply(get_state)
sm_long['state2'] = sm_long['celltype2'].apply(get_state)

# 동일 state pair만 사용 (Healthy-Healthy / MASL-MASL)
sm_healthy = sm_long.query('state1 == "Healthy" and state2 == "Healthy"').copy()
sm_masl    = sm_long.query('state1 == "MASL"    and state2 == "MASL"').copy()

for df_ in [sm_healthy, sm_masl]:
    for c in ['celltype1', 'celltype2']:
        df_[c] = df_[c].apply(clean_name)

print('Healthy pairs:', sm_healthy.shape[0])
print('MASL    pairs:', sm_masl.shape[0])

In [ ]:
# ── 전체 cluster-cluster delta summary ──────────────────────────────────────
healthy_summary_cc = (
    sm_healthy.groupby(['celltype1', 'celltype2'])['weight'].mean().reset_index()
)
masl_summary_cc = (
    sm_masl.groupby(['celltype1', 'celltype2'])['weight'].mean().reset_index()
)

delta_cc = pd.merge(
    masl_summary_cc, healthy_summary_cc,
    on=['celltype1', 'celltype2'],
    suffixes=('_MASL', '_Healthy'),
    how='outer'
).fillna(0)
delta_cc['delta_weight'] = delta_cc['weight_MASL'] - delta_cc['weight_Healthy']

# 클러스터-클러스터 delta heatmap
pivot_delta_cc = delta_cc.pivot_table(
    index='celltype1', columns='celltype2',
    values='delta_weight', fill_value=0
)

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    pivot_delta_cc,
    cmap='RdBu_r', center=0,
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'Δ mean expression (MASL − Healthy)'},
    ax=ax
)
ax.set_xlabel('Receiver cluster', fontsize=12)
ax.set_ylabel('Sender cluster',   fontsize=12)
ax.set_title('Δ Cluster-Cluster Interaction (MASL − Healthy)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_delta_cc_heatmap.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_delta_cc_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

delta_cc.to_csv(os.path.join(OUT_DIR, 'delta_cc_masl_healthy.csv'), index=False)
print('delta_cc_masl_healthy.csv 저장 완료')

## 7. Mono3 Source / Target Pivot 분석

`significant_means`에서 mono3가 **sender(source)** 또는 **receiver(target)**인 컬럼만 필터링해  
MASL − Healthy delta를 계산하고 interaction pair × partner cluster pivot heatmap을 그린다.

In [ ]:
# ── mono3 sender(source) 분석 ────────────────────────────────────────────────
mono3_src_cols = [c for c in sm_pair_cols if 'mono3' in c.split('|')[0]]
print(f'mono3 as sender: {len(mono3_src_cols)} cluster pairs')

src_long = sig_means[['interacting_pair'] + mono3_src_cols].melt(
    id_vars='interacting_pair', var_name='pair', value_name='weight'
).dropna(subset=['weight'])

src_long[['celltype1', 'celltype2']] = src_long['pair'].str.split('|', expand=True)
src_long['state1'] = src_long['celltype1'].apply(get_state)
src_long['state2'] = src_long['celltype2'].apply(get_state)
for c in ['celltype1', 'celltype2']:
    src_long[c] = src_long[c].apply(clean_name)

src_healthy = src_long.query('state1 == "Healthy" and state2 == "Healthy"')
src_masl    = src_long.query('state1 == "MASL"    and state2 == "MASL"')

src_h_sum = src_healthy.groupby(['interacting_pair', 'celltype2'])['weight'].mean().reset_index()
src_m_sum = src_masl.groupby(['interacting_pair', 'celltype2'])['weight'].mean().reset_index()

delta_src = pd.merge(
    src_m_sum, src_h_sum,
    on=['interacting_pair', 'celltype2'],
    suffixes=('_MASL', '_Healthy'), how='outer'
).fillna(0)
delta_src['delta_weight'] = delta_src['weight_MASL'] - delta_src['weight_Healthy']

pivot_src = delta_src.pivot_table(
    index='interacting_pair', columns='celltype2',
    values='delta_weight', fill_value=0
)

# 행 변화량 기준 필터 (|delta| 상위 행만 표시)
row_max = pivot_src.abs().max(axis=1)
thr_src = row_max.quantile(0.7)
pivot_src_plot = pivot_src[row_max >= thr_src]

fig_h = max(6, 0.35 * len(pivot_src_plot))
fig, ax = plt.subplots(figsize=(12, fig_h))
sns.heatmap(
    pivot_src_plot, cmap='RdBu_r', center=0,
    linewidths=0.4, linecolor='lightgray',
    cbar_kws={'label': 'Δ weight (MASL − Healthy)'},
    ax=ax
)
ax.set_title('mono3 as SENDER — top Δ LR interactions (MASL − Healthy)', fontsize=12)
ax.set_xlabel('Receiver cluster')
ax.set_ylabel('Interacting pair')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_mono3_source_delta.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_mono3_source_delta.png'), dpi=150, bbox_inches='tight')
plt.show()

pivot_src.to_csv(os.path.join(OUT_DIR, 'mono3_source_delta.csv'))
print('mono3_source_delta.csv 저장 완료')

In [ ]:
# ── mono3 receiver(target) 분석 ──────────────────────────────────────────────
mono3_tgt_cols = [c for c in sm_pair_cols if 'mono3' in c.split('|')[1]]
print(f'mono3 as receiver: {len(mono3_tgt_cols)} cluster pairs')

tgt_long = sig_means[['interacting_pair'] + mono3_tgt_cols].melt(
    id_vars='interacting_pair', var_name='pair', value_name='weight'
).dropna(subset=['weight'])

tgt_long[['celltype1', 'celltype2']] = tgt_long['pair'].str.split('|', expand=True)
tgt_long['state1'] = tgt_long['celltype1'].apply(get_state)
tgt_long['state2'] = tgt_long['celltype2'].apply(get_state)
for c in ['celltype1', 'celltype2']:
    tgt_long[c] = tgt_long[c].apply(clean_name)

tgt_healthy = tgt_long.query('state1 == "Healthy" and state2 == "Healthy"')
tgt_masl    = tgt_long.query('state1 == "MASL"    and state2 == "MASL"')

tgt_h_sum = tgt_healthy.groupby(['interacting_pair', 'celltype1'])['weight'].mean().reset_index()
tgt_m_sum = tgt_masl.groupby(['interacting_pair', 'celltype1'])['weight'].mean().reset_index()

delta_tgt = pd.merge(
    tgt_m_sum, tgt_h_sum,
    on=['interacting_pair', 'celltype1'],
    suffixes=('_MASL', '_Healthy'), how='outer'
).fillna(0)
delta_tgt['delta_weight'] = delta_tgt['weight_MASL'] - delta_tgt['weight_Healthy']

pivot_tgt = delta_tgt.pivot_table(
    index='interacting_pair', columns='celltype1',
    values='delta_weight', fill_value=0
)

row_max_t = pivot_tgt.abs().max(axis=1)
thr_tgt = row_max_t.quantile(0.7)
pivot_tgt_plot = pivot_tgt[row_max_t >= thr_tgt]

fig_h = max(6, 0.35 * len(pivot_tgt_plot))
fig, ax = plt.subplots(figsize=(12, fig_h))
sns.heatmap(
    pivot_tgt_plot, cmap='RdBu_r', center=0,
    linewidths=0.4, linecolor='lightgray',
    cbar_kws={'label': 'Δ weight (MASL − Healthy)'},
    ax=ax
)
ax.set_title('mono3 as RECEIVER — top Δ LR interactions (MASL − Healthy)', fontsize=12)
ax.set_xlabel('Sender cluster')
ax.set_ylabel('Interacting pair')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_mono3_target_delta.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_mono3_target_delta.png'), dpi=150, bbox_inches='tight')
plt.show()

pivot_tgt.to_csv(os.path.join(OUT_DIR, 'mono3_target_delta.csv'))
print('mono3_target_delta.csv 저장 완료')

In [ ]:
# ── mono3 top-N up/down 상호작용 bar chart ───────────────────────────────────
TOP_N = 15

# sender 관점 top/bottom
src_top = delta_src.sort_values('delta_weight', ascending=False).head(TOP_N)
src_bot = delta_src.sort_values('delta_weight').head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax_, df_, title_, color_ in [
    (axes[0], src_top, f'mono3 sender — top {TOP_N} UP in MASL',   'firebrick'),
    (axes[1], src_bot, f'mono3 sender — top {TOP_N} DOWN in MASL', 'steelblue'),
]:
    label = df_['interacting_pair'] + ' → ' + df_['celltype2']
    ax_.barh(label, df_['delta_weight'].abs(), color=color_)
    ax_.set_title(title_, fontsize=10)
    ax_.set_xlabel('|Δ weight|')
    ax_.invert_yaxis()
    ax_.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_mono3_source_top_interactions.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(FIG_DIR, 'cpdb_method3_mono3_source_top_interactions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Classification 기반 필터 분석

`significant_means`의 `classification` 컬럼으로 Chemokine / Interleukin signaling을 분리해  
mono3 관련 상호작용만 추출한다.

In [ ]:
# ── classification 분포 확인 ─────────────────────────────────────────────────
print('classification 종류:')
print(sig_means['classification'].value_counts().head(20))

In [ ]:
# ── Chemokine / Interleukin signaling 필터 ────────────────────────────────────
chemo_df = sig_means[
    sig_means['classification'].str.contains('Chemokine', case=False, na=False)
].copy()
il_df = sig_means[
    sig_means['classification'].str.contains('Interleukin', case=False, na=False)
].copy()

print(f'Chemokine signaling LR pairs: {len(chemo_df)}')
print(f'Interleukin signaling LR pairs: {len(il_df)}')

chemo_genes = set(chemo_df['gene_a'].dropna().tolist() + chemo_df['gene_b'].dropna().tolist())
il_genes    = set(il_df['gene_a'].dropna().tolist() + il_df['gene_b'].dropna().tolist())

print(f'Chemokine 관련 유전자: {len(chemo_genes)}')
print(f'Interleukin 관련 유전자: {len(il_genes)}')

In [ ]:
# ── mono3 관련 Chemokine LR pairs — delta pivot ──────────────────────────────
for label_, class_df in [('Chemokine', chemo_df), ('Interleukin', il_df)]:
    class_pairs = set(class_df['interacting_pair'].tolist())

    # sender 관점
    d_src_cls = delta_src[delta_src['interacting_pair'].isin(class_pairs)].copy()
    if len(d_src_cls) == 0:
        print(f'[{label_}] mono3 sender: 해당 없음')
        continue

    p = d_src_cls.pivot_table(
        index='interacting_pair', columns='celltype2',
        values='delta_weight', fill_value=0
    )
    fig_h = max(4, 0.4 * len(p))
    fig, ax = plt.subplots(figsize=(12, fig_h))
    sns.heatmap(
        p, cmap='RdBu_r', center=0,
        linewidths=0.4, linecolor='lightgray',
        cbar_kws={'label': 'Δ weight (MASL − Healthy)'},
        ax=ax
    )
    ax.set_title(f'mono3 sender — {label_} signaling Δ (MASL − Healthy)', fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    fname = f'cpdb_method3_mono3_source_{label_.lower()}_delta'
    plt.savefig(os.path.join(FIG_DIR, fname + '.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(FIG_DIR, fname + '.png'), dpi=150, bbox_inches='tight')
    plt.show()
    p.to_csv(os.path.join(OUT_DIR, fname + '.csv'))
    print(f'{fname}.csv 저장 완료')

## 9. kpy.plot_cpdb Dot Plot (Healthy vs MASL)

`ktplotspy.plot_cpdb()`로 mono3-Healthy / mono3-MASL 대상 interaction dot plot을 그린다.  
유전자 목록은 상위 delta LR pair에서 자동 추출하거나 직접 지정한다.

In [ ]:
# ── dot plot용 top gene 자동 선택 ────────────────────────────────────────────
TOP_GENES_N = 20

top_src_genes = set()
for lr in delta_src.nlargest(TOP_GENES_N, 'delta_weight')['interacting_pair']:
    row = sig_means[sig_means['interacting_pair'] == lr]
    if len(row):
        top_src_genes.update(row[['gene_a', 'gene_b']].dropna().values.flatten())

top_src_genes = [g for g in top_src_genes if isinstance(g, str) and g != 'nan']
print(f'dot plot 대상 유전자 ({len(top_src_genes)}개):')
print(top_src_genes[:15])

In [ ]:
# ── mono3-Healthy dot plot ────────────────────────────────────────────────────
if 'X_umap' in adata.obsm and top_src_genes:
    from plotnine import facet_wrap

    # partner cluster 목록 (Healthy state)
    healthy_partners = '|'.join([
        c for c in adata.obs[CELLTYPE_COL].unique()
        if 'Healthy' in c and 'mono3' not in c
    ])

    if healthy_partners:
        p = kpy.plot_cpdb(
            adata         = adata,
            cell_type1    = 'mono3-Healthy',
            cell_type2    = healthy_partners,
            means         = cpdb_results['means'],
            pvals         = cpdb_results['relevant_interactions'],
            celltype_key  = CELLTYPE_COL,
            genes         = top_src_genes[:15],
            figsize       = (14, 8),
            title         = 'mono3-Healthy interactions (top delta genes)',
            max_size      = 6,
            highlight_size= 0.75,
            degs_analysis = True,
            standard_scale= True
        )
        fig_out = p + facet_wrap('~ classification', ncol=3)
        fig_out.save(
            os.path.join(FIG_DIR, 'cpdb_method3_mono3_healthy_dotplot.pdf'),
            format='pdf', bbox_inches='tight'
        )
        print(fig_out)
    else:
        print('[주의] Healthy partner cluster 없음')
else:
    print('[주의] UMAP 좌표 없거나 gene 목록 없음 — dot plot 건너뜀')

In [ ]:
# ── mono3-MASL dot plot ───────────────────────────────────────────────────────
if 'X_umap' in adata.obsm and top_src_genes:
    from plotnine import facet_wrap

    masl_partners = '|'.join([
        c for c in adata.obs[CELLTYPE_COL].unique()
        if 'MASL' in c and 'mono3' not in c
    ])

    if masl_partners:
        p = kpy.plot_cpdb(
            adata         = adata,
            cell_type1    = 'mono3-MASL',
            cell_type2    = masl_partners,
            means         = cpdb_results['means'],
            pvals         = cpdb_results['relevant_interactions'],
            celltype_key  = CELLTYPE_COL,
            genes         = top_src_genes[:15],
            figsize       = (14, 8),
            title         = 'mono3-MASL interactions (top delta genes)',
            max_size      = 6,
            highlight_size= 0.75,
            degs_analysis = True,
            standard_scale= True
        )
        fig_out = p + facet_wrap('~ classification', ncol=3)
        fig_out.save(
            os.path.join(FIG_DIR, 'cpdb_method3_mono3_masl_dotplot.pdf'),
            format='pdf', bbox_inches='tight'
        )
        print(fig_out)
    else:
        print('[주의] MASL partner cluster 없음')
else:
    print('[주의] UMAP 좌표 없거나 gene 목록 없음 — dot plot 건너뜀')

## 10. Per-Cell Score 테이블 CSV 저장

In [ ]:
# ── per-cell score 컬럼만 추출해서 CSV 저장 ───────────────────────────────────
# barcode, cluster_state, 모든 CCI score 컬럼
save_cols = [CELLTYPE_COL] + cci_cols
score_export = adata.obs[save_cols].copy()
score_export.index.name = 'barcode'

score_export.to_csv(SCORE_CSV_OUT)
print(f'저장 완료: {SCORE_CSV_OUT}')
print(f'  shape: {score_export.shape}')
score_export.head(3)

## 11. adata 저장

In [ ]:
# ── CCI score가 포함된 adata 저장 ─────────────────────────────────────────────
# obs에 추가된 컬럼 수 확인
n_cci = len([c for c in adata.obs.columns if c.endswith('__score') or c.startswith('CCI_')])
print(f'저장할 adata obs 컬럼 중 CCI 관련: {n_cci}개')
print(f'저장 경로: {ADATA_OUT}')

adata.write_h5ad(ADATA_OUT)
print('저장 완료!')

In [ ]:
# ── 최종 결과 요약 ─────────────────────────────────────────────────────────────
print('=' * 60)
print('[ 분석 완료 요약 ]')
print('=' * 60)
print(f'adata 세포 수:              {adata.n_obs}')
print(f'유의 LR pair 수:            {len(sig_lr_pairs)}')
print(f'추가된 per-cell 컬럼 수:    {len(cci_cols)}')
print()
print('저장된 파일:')
print(f'  {PKL_PATH}')
print(f'  {ADATA_OUT}')
print(f'  {SCORE_CSV_OUT}')
print(f'  {os.path.join(OUT_DIR, "cluster_pair_total_score.csv")}')
print()
print('컬럼명 규칙: {sender_cluster}__{LR_pair}__score')
print('예시:', [c for c in cci_cols if c.endswith('__score')][:3])